In [1]:
#!/usr/bin/env python3
"""
================================================
 SLR Network Analysis Tool v2.0 - ENHANCED
================================================
Systematic Literature Review con Network Analysis,
Gap Analysis Integration, e Publication-Quality Visualizations.

Features:
- Multi-API fetching (arXiv, OpenAlex, S2, Crossref) con resilient retry
- Fuzzy deduplication
- Metadata enrichment (TRL, validation type, domain, standards)
- Dual-mode visualization (PNG high-res + HTML interactive)
- Gap-aware analytics (domain matrix, TRL distribution, standards adoption)
- Export multipli (CSV, LaTeX, BibTeX, GEXF)

Author: Enhanced from original SLR tool
Date: 2025
License: MIT
"""

import os
import re
import json
import time
import html
import math
import itertools
import random
import logging
import argparse
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple
import xml.etree.ElementTree as ET

import requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patheffects import withStroke, Normal
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    import community as community_louvain
except ImportError:
    raise ImportError("Install 'python-louvain': pip install python-louvain")

try:
    from pyvis.network import Network
    HAS_PYVIS = True
except ImportError:
    HAS_PYVIS = False
    logging.warning("PyVis not available: HTML interactive output will be skipped")

try:
    from fuzzywuzzy import fuzz
    HAS_FUZZY = True
except ImportError:
    HAS_FUZZY = False
    logging.warning("FuzzyWuzzy not available: using exact matching only")

try:
    import yaml
    HAS_YAML = True
except ImportError:
    HAS_YAML = False
    logging.warning("PyYAML not available: using hardcoded config")

# ===========================
# LOGGING SETUP
# ===========================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('slr_network.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ===========================
# DEFAULT CONFIGURATION
# ===========================
DEFAULT_CONFIG = {
    # API Parameters
    'max_per_keyword': 40,
    'years_back': 10,
    'respectful_delay': 0.5,
    'max_retries': 4,
    'cache_dir': 'data_cache_slr',
    
    # Network Parameters
    'sim_threshold': 0.28,
    'min_degree': 2,
    'seed': 42,
    
    # Visualization
    'figsize': (16, 12),
    'png_dpi': 300,
    'html_height': '860px',
    'html_width': '100%',
    
    # Keywords
    'keywords': [
        "predictive maintenance",
        "synthetic data",
        "digital twin",
        "condition monitoring",
        "reliability engineering"
    ],
    
    # Publishers Priority
    'top_publishers': [
        "Elsevier", "Wiley", "Springer", "Springer Nature",
        "IEEE", "Association for Computing Machinery", "ACM",
        "MDPI", "Taylor & Francis", "Informa UK Limited",
        "American Society of Civil Engineers", "ASCE"
    ],
    
    # Color Palette (Modern Viridis-inspired)
    'topic_colors': {
        "predictive maintenance": "#d62728",  # Red - urgency
        "synthetic data": "#2ca02c",          # Green - generation
        "digital twin": "#1f77b4",            # Blue - virtual
        "condition monitoring": "#ff7f0e",    # Orange - alert
        "reliability engineering": "#9467bd"  # Purple - rigor
    },
    
    # Gap Analysis Keywords
    'domain_keywords': {
        'HVAC': ['hvac', 'air conditioning', 'chiller', 'ahu', 'air handling'],
        'Manufacturing': ['manufacturing', 'production', 'assembly', 'cnc', 'machine tool'],
        'Energy': ['wind turbine', 'solar', 'battery', 'grid', 'power plant'],
        'Infrastructure': ['bridge', 'railway', 'pipeline', 'building', 'road']
    },
    
    'validation_keywords': {
        'deployed': ['industrial implementation', 'operational', 'field deployment', 'production system'],
        'case_study': ['case study', 'real-world', 'industrial case'],
        'benchmark': ['benchmark', 'experimental', 'controlled environment'],
        'simulation': ['simulation', 'simulated', 'synthetic environment']
    },
    
    'standard_patterns': [
        r'ISO[\s-]?13374',
        r'ISO[\s-]?23247',
        r'OSA-CBM',
        r'MIMOSA',
        r'IEC[\s-]?61850'
    ],
    
    'dataset_keywords': ['dataset', 'github', 'repository', 'zenodo', 'figshare', 'data availability']
}

# ===========================
# CONFIGURATION LOADER
# ===========================
def load_config(config_file: Optional[str] = 'config.yaml') -> Dict[str, Any]:
    """Load configuration from YAML file or use defaults"""
    if config_file and HAS_YAML and Path(config_file).exists():
        try:
            with open(config_file, 'r') as f:
                user_config = yaml.safe_load(f)
            config = {**DEFAULT_CONFIG, **user_config}
            logger.info(f"Loaded configuration from {config_file}")
            return config
        except Exception as e:
            logger.warning(f"Failed to load config file: {e}. Using defaults.")
    
    return DEFAULT_CONFIG.copy()

# Global config (will be initialized in main)
CONFIG = DEFAULT_CONFIG.copy()
# ── API KEY: Semantic Scholar ────────────────────────────────────────────────
os.environ['SEMANTICSCHOLAR_API'] = 'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXxxx' # <- inserthere your API
# ===========================
# UTILITY FUNCTIONS
# ===========================
def _cache_path(key: str, suffix: str = "json") -> str:
    """Generate safe cache file path"""
    safe = re.sub(r"[^a-zA-Z0-9_.-]", "_", key)[:200]  # Truncate long keys
    return os.path.join(CONFIG['cache_dir'], f"{safe}.{suffix}")

def clean_text(text: str) -> str:
    """Clean and normalize text"""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def safe_int(value: Any, default: int = 0) -> int:
    """Safely convert to int with default"""
    try:
        if value is None or (isinstance(value, float) and np.isnan(value)):
            return default
        return int(value)
    except (ValueError, TypeError):
        return default

def match_keywords(text: str, keywords: List[str]) -> List[str]:
    """Find which keywords are present in text (word-boundary aware)"""
    text_lower = " " + text.lower() + " "
    found = []
    for kw in keywords:
        kw_lower = kw.lower()
        pattern = r'\b' + re.escape(kw_lower) + r'\b'
        if re.search(pattern, text_lower):
            found.append(kw)
    return sorted(set(found))

# ===========================
# API ERROR HANDLING
# ===========================
class APIError(Exception):
    """Custom exception for API failures"""
    pass

def resilient_http_get(url: str, params: Optional[Dict] = None, 
                       headers: Optional[Dict] = None, 
                       expect: str = "json") -> Any:
    """HTTP GET with exponential backoff retry"""
    headers = headers or {"User-Agent": "SLR-Network/2.0 (Academic Research Tool)"}
    delay = CONFIG['respectful_delay']
    
    for attempt in range(CONFIG['max_retries']):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=40)
            
            if response.status_code == 200:
                return response.json() if expect == "json" else response.text
            
            if response.status_code in (429, 500, 502, 503, 504):
                logger.warning(f"Retry {attempt+1}/{CONFIG['max_retries']} for {url}: Status {response.status_code}")
                time.sleep(delay)
                delay *= 2
                continue
            
            response.raise_for_status()
            
        except requests.exceptions.RequestException as e:
            if attempt == CONFIG['max_retries'] - 1:
                raise APIError(f"Failed after {CONFIG['max_retries']} attempts: {e}")
            logger.warning(f"Request error (attempt {attempt+1}): {e}")
            time.sleep(delay)
            delay *= 2
    
    raise APIError(f"Max retries exceeded for {url}")

def fetch_json_cached(url: str, params: Optional[Dict] = None, 
                     headers: Optional[Dict] = None, force: bool = False) -> Dict:
    """Fetch JSON with caching"""
    key = url
    if params:
        key += "?" + "&".join(f"{k}={v}" for k, v in sorted(params.items()))
    
    cache_file = _cache_path(key, "json")
    
    if not force and os.path.exists(cache_file):
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception as e:
            logger.warning(f"Cache read error: {e}")
    
    data = resilient_http_get(url, params=params, headers=headers, expect="json")
    
    try:
        with open(cache_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    except Exception as e:
        logger.warning(f"Cache write error: {e}")
    
    time.sleep(CONFIG['respectful_delay'])
    return data

def fetch_text_cached(url: str, params: Optional[Dict] = None, 
                     headers: Optional[Dict] = None, force: bool = False) -> str:
    """Fetch text with caching"""
    key = url
    if params:
        key += "?" + "&".join(f"{k}={v}" for k, v in sorted(params.items()))
    
    cache_file = _cache_path(key, "txt")
    
    if not force and os.path.exists(cache_file):
        try:
            with open(cache_file, 'r', encoding='utf-8') as f:
                return f.read()
        except Exception as e:
            logger.warning(f"Cache read error: {e}")
    
    text = resilient_http_get(url, params=params, headers=headers, expect="text")
    
    try:
        with open(cache_file, 'w', encoding='utf-8') as f:
            f.write(text)
    except Exception as e:
        logger.warning(f"Cache write error: {e}")
    
    time.sleep(CONFIG['respectful_delay'])
    return text

# ===========================
# OPENALEX UTILITIES
# ===========================
def openalex_reconstruct_abstract(inverted_index: Dict) -> str:
    """Reconstruct abstract from OpenAlex inverted index"""
    if not isinstance(inverted_index, dict) or not inverted_index:
        return ""
    
    try:
        max_pos = max(max(positions) for positions in inverted_index.values())
        words = [""] * (max_pos + 1)
        
        for word, positions in inverted_index.items():
            for pos in positions:
                if 0 <= pos < len(words):
                    words[pos] = word
        
        return clean_text(" ".join(word for word in words if word))
    except Exception as e:
        logger.warning(f"Failed to reconstruct OpenAlex abstract: {e}")
        return ""

# Continue in next file part...

In [2]:
# ===========================
# PART 2: API SEARCH FUNCTIONS
# ===========================

def search_arxiv(query: str, max_results: Optional[int] = None, 
                 years_back: Optional[int] = None) -> List[Dict[str, Any]]:
    """Search arXiv for papers
    
    Args:
        query: Search query string
        max_results: Maximum papers to return
        years_back: Only papers from last N years
        
    Returns:
        List of paper dictionaries
    """
    max_results = max_results or CONFIG['max_per_keyword']
    years_back = years_back or CONFIG['years_back']
    
    base_url = "http://export.arxiv.org/api/query"
    year_cutoff = datetime.now(timezone.utc).year - years_back
    
    params = {
        'search_query': f'all:"{query}"',
        'start': 0,
        'max_results': max_results,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending'
    }
    
    try:
        xml_text = fetch_text_cached(base_url, params=params)
        root = ET.fromstring(xml_text)
    except Exception as e:
        logger.error(f"arXiv API error for '{query}': {e}")
        return []
    
    namespace = {'atom': 'http://www.w3.org/2005/Atom'}
    results = []
    
    for entry in root.findall('atom:entry', namespace):
        title = entry.findtext('atom:title', default='', namespaces=namespace)
        abstract = entry.findtext('atom:summary', default='', namespaces=namespace)
        published = entry.findtext('atom:published', default='', namespaces=namespace)
        
        year = None
        if published:
            try:
                year = int(published[:4])
                if year < year_cutoff:
                    continue
            except ValueError:
                pass
        
        authors = [
            author.findtext('atom:name', namespaces=namespace) 
            for author in entry.findall('atom:author', namespace)
        ]
        
        pdf_url = None
        for link in entry.findall('atom:link', namespace):
            if link.attrib.get('type') == 'application/pdf':
                pdf_url = link.attrib.get('href')
                break
        
        results.append({
            'id': entry.findtext('atom:id', default='', namespaces=namespace),
            'doi': None,
            'title': clean_text(title),
            'abstract': clean_text(abstract),
            'authors': [a for a in authors if a],
            'venue': 'arXiv',
            'publisher': 'arXiv',
            'year': year,
            'url': pdf_url,
            'source': 'arXiv',
            'is_oa': True,
            'cited_by': None
        })
    
    logger.info(f"arXiv: {len(results)} papers for '{query}'")
    return results


def search_openalex(query: str, per_page: int = 50, max_pages: int = 1,
                   years_back: Optional[int] = None) -> List[Dict[str, Any]]:
    """Search OpenAlex for papers
    
    Args:
        query: Search query string
        per_page: Results per page
        max_pages: Maximum pages to fetch
        years_back: Only papers from last N years
        
    Returns:
        List of paper dictionaries
    """
    years_back = years_back or CONFIG['years_back']
    base_url = "https://api.openalex.org/works"
    year_cutoff = datetime.now(timezone.utc).year - years_back
    
    results = []
    
    for page in range(1, max_pages + 1):
        params = {
            'search': query,
            'filter': f'from_publication_date:{year_cutoff}-01-01,is_paratext:false',
            'per_page': per_page,
            'page': page,
            'sort': 'cited_by_count:desc'
        }
        
        try:
            data = fetch_json_cached(base_url, params=params)
        except Exception as e:
            logger.error(f"OpenAlex API error for '{query}' page {page}: {e}")
            break
        
        for work in data.get('results', []):
            # Extract abstract
            abstract_text = ""
            if isinstance(work.get('abstract'), str):
                abstract_text = clean_text(work['abstract'])
            elif isinstance(work.get('abstract_inverted_index'), dict):
                abstract_text = openalex_reconstruct_abstract(work['abstract_inverted_index'])
            
            # Extract authors
            authors = [
                authorship.get('author', {}).get('display_name')
                for authorship in work.get('authorships', [])
            ]
            
            # Extract venue info
            host_venue = work.get('host_venue', {}) or {}
            primary_location = work.get('primary_location', {}) or {}
            
            results.append({
                'id': work.get('id'),
                'doi': (work.get('doi') or '').lower(),
                'title': clean_text(work.get('title', '')),
                'abstract': abstract_text,
                'authors': [a for a in authors if a],
                'venue': host_venue.get('display_name'),
                'publisher': host_venue.get('publisher'),
                'year': work.get('publication_year'),
                'url': (work.get('open_access', {}) or {}).get('oa_url') or primary_location.get('landing_page_url'),
                'source': 'OpenAlex',
                'is_oa': bool((work.get('open_access', {}) or {}).get('is_oa', False)),
                'cited_by': work.get('cited_by_count', 0)
            })
    
    logger.info(f"OpenAlex: {len(results)} papers for '{query}'")
    return results


def search_semanticscholar(query: str, limit: Optional[int] = None,
                          years_back: Optional[int] = None,
                          api_key: Optional[str] = None) -> List[Dict[str, Any]]:
    """Search Semantic Scholar for papers
    
    Args:
        query: Search query string
        limit: Maximum papers to return
        years_back: Only papers from last N years
        api_key: Semantic Scholar API key (optional, from env or param)
        
    Returns:
        List of paper dictionaries
    """
    limit = limit or CONFIG['max_per_keyword']
    years_back = years_back or CONFIG['years_back']
    
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    year_cutoff = datetime.now(timezone.utc).year - years_back
    
    headers = {'User-Agent': 'SLR-Network/2.0 (Academic Research)'}
    
    # Try to get API key
    api_key = api_key or os.environ.get('SEMANTICSCHOLAR_API')
    if api_key and api_key != 'YOUR_S2_API_KEY_PLACEHOLDER':
        headers['x-api-key'] = api_key
    
    params = {
        'query': query,
        'limit': limit,
        'fields': 'title,abstract,year,authors,url,venue,citationCount,externalIds,journal'
    }
    
    try:
        data = fetch_json_cached(base_url, params=params, headers=headers)
    except Exception as e:
        logger.warning(f"Semantic Scholar API error for '{query}': {e}")
        return []
    
    results = []
    
    for paper in data.get('data', []):
        year = paper.get('year')
        if year and year < year_cutoff:
            continue
        
        authors = [
            author.get('name')
            for author in paper.get('authors', [])
            if author.get('name')
        ]
        
        doi = ((paper.get('externalIds', {}) or {}).get('DOI', '') or '').lower()
        
        publisher = None
        journal_info = paper.get('journal')
        if isinstance(journal_info, dict):
            publisher = journal_info.get('publisher')
        
        results.append({
            'id': paper.get('paperId'),
            'doi': doi,
            'title': clean_text(paper.get('title', '')),
            'abstract': clean_text(paper.get('abstract', '')),
            'authors': authors,
            'venue': paper.get('venue'),
            'publisher': publisher,
            'year': year,
            'url': paper.get('url'),
            'source': 'SemanticScholar',
            'is_oa': True,  # S2 provides open metadata
            'cited_by': paper.get('citationCount')
        })
    
    logger.info(f"Semantic Scholar: {len(results)} papers for '{query}'")
    return results


def search_crossref_publishers(query: str, rows: Optional[int] = None,
                               years_back: Optional[int] = None,
                               publishers: Optional[List[str]] = None) -> List[Dict[str, Any]]:
    """Search Crossref for papers from top publishers
    
    Args:
        query: Search query string
        rows: Maximum papers to return
        years_back: Only papers from last N years
        publishers: List of publisher names to filter
        
    Returns:
        List of paper dictionaries
    """
    rows = rows or CONFIG['max_per_keyword']
    years_back = years_back or CONFIG['years_back']
    publishers = publishers or CONFIG['top_publishers']
    
    base_url = "https://api.crossref.org/works"
    year_cutoff = datetime.now(timezone.utc).year - years_back
    
    # Fixed: proper filter formatting for Crossref API
    filter_parts = [
        "type:journal-article",
        f"from-pub-date:{year_cutoff}-01-01"
    ]
    
    params = {
        'query': query,
        'rows': rows,
        'filter': ','.join(filter_parts),  # Proper comma joining
        'sort': 'is-referenced-by-count',
        'order': 'desc',
        # Simplified select to avoid malformed requests
        'select': 'DOI,title,author,issued,URL,container-title,is-referenced-by-count,publisher'
    }
    
    headers = {
        'User-Agent': 'SLR-Network/2.0 (Academic Research; mailto:research@example.org)'
    }
    
    try:
        data = fetch_json_cached(base_url, params=params, headers=headers)
    except Exception as e:
        logger.warning(f"Crossref API error for '{query}': {e}")
        return []
    
    results = []
    
    for item in (data.get('message', {}) or {}).get('items', []):
        publisher_name = item.get('publisher', '')
        
        # Filter by publisher
        if not any(pub.lower() in publisher_name.lower() for pub in publishers):
            continue
        
        # Extract year
        year = None
        issued = item.get('issued', {}).get('date-parts', [])
        if issued and issued[0]:
            year = issued[0][0]
        
        # Extract title
        title_list = item.get('title', [])
        title = title_list[0] if title_list else ''
        
        # Extract authors
        authors = []
        for author in item.get('author', []) or []:
            given = author.get('given', '')
            family = author.get('family', '')
            name = f"{given} {family}".strip()
            if name:
                authors.append(name)
        
        # Extract venue
        venue_list = item.get('container-title', [])
        venue = venue_list[0] if venue_list else ''
        
        results.append({
            'id': item.get('DOI'),
            'doi': (item.get('DOI') or '').lower(),
            'title': clean_text(title),
            'abstract': '',  # Crossref typically doesn't include abstracts
            'authors': authors,
            'venue': venue,
            'publisher': publisher_name,
            'year': year,
            'url': item.get('URL'),
            'source': 'Crossref',
            'is_oa': None,  # Not directly available
            'cited_by': item.get('is-referenced-by-count', 0)
        })
    
    logger.info(f"Crossref: {len(results)} papers for '{query}' (from target publishers)")
    return results


# ===========================
# DEDUPLICATION
# ===========================

def exact_dedup(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Deduplicate by DOI or exact title match"""
    by_key = {}
    
    for record in records:
        # Normalize cited_by
        record['cited_by'] = safe_int(record.get('cited_by'), 0)
        
        # Generate key: DOI if available, else lowercase title
        key = record.get('doi') or clean_text(record.get('title', '')).lower()
        
        if not key:
            continue
        
        if key in by_key:
            current = by_key[key]
            # Keep record with better data
            better = False
            
            # Prefer records with abstract
            if record.get('abstract') and not current.get('abstract'):
                better = True
            # Or higher citation count
            elif record['cited_by'] > safe_int(current.get('cited_by')):
                better = True
            
            if better:
                by_key[key] = record
        else:
            by_key[key] = record
    
    return list(by_key.values())


def fuzzy_dedup(records: List[Dict[str, Any]], threshold: int = 90) -> List[Dict[str, Any]]:
    """Deduplicate using fuzzy title matching (requires fuzzywuzzy)
    
    Args:
        records: List of paper records
        threshold: Fuzzy match threshold (0-100)
        
    Returns:
        Deduplicated list
    """
    if not HAS_FUZZY:
        logger.warning("FuzzyWuzzy not available, using exact dedup only")
        return exact_dedup(records)
    
    by_key = {}
    
    for record in records:
        record['cited_by'] = safe_int(record.get('cited_by'), 0)
        
        doi = record.get('doi')
        title_lower = clean_text(record.get('title', '')).lower()
        
        if not title_lower:
            continue
        
        # If has DOI, use exact matching
        if doi:
            key = doi
            matched = False
        else:
            # Check fuzzy match against existing titles
            key = title_lower
            matched = False
            
            for existing_key in list(by_key.keys()):
                # Skip DOI keys
                if existing_key.startswith('10.'):
                    continue
                
                ratio = fuzz.ratio(title_lower, existing_key)
                
                if ratio > threshold:
                    matched = True
                    key = existing_key
                    
                    # Merge logic: keep better record
                    current = by_key[key]
                    if record.get('abstract') and not current.get('abstract'):
                        by_key[key] = record
                    elif record['cited_by'] > safe_int(current.get('cited_by')):
                        by_key[key] = record
                    
                    break
        
        if not matched:
            by_key[key] = record
    
    return list(by_key.values())


# Continue in next part...

In [3]:
# ===========================
# PART 3: METADATA ENRICHMENT (Gap Analysis Integration)
# ===========================

def classify_validation_type(text: str) -> str:
    """Classify paper validation approach
    
    Categories:
    - deployed: Industrial/operational deployment
    - case_study: Real-world case study
    - benchmark: Experimental benchmark
    - simulation: Simulation-based
    - unknown: Cannot determine
    """
    text_lower = text.lower()
    
    for vtype, keywords in CONFIG['validation_keywords'].items():
        if any(kw in text_lower for kw in keywords):
            return vtype
    
    return 'unknown'


def classify_application_domain(text: str) -> str:
    """Classify application domain
    
    Categories:
    - HVAC, Manufacturing, Energy, Infrastructure
    - Multi: Multiple domains
    - General: Not domain-specific
    """
    text_lower = text.lower()
    matches = []
    
    for domain, keywords in CONFIG['domain_keywords'].items():
        if any(kw in text_lower for kw in keywords):
            matches.append(domain)
    
    if len(matches) == 0:
        return 'General'
    elif len(matches) == 1:
        return matches[0]
    else:
        return 'Multi'


def estimate_trl(row: pd.Series) -> int:
    """Estimate Technology Readiness Level (TRL 1-9)
    
    Heuristics:
    - deployed validation → TRL 8-9
    - case_study → TRL 6-7
    - benchmark → TRL 4-5
    - simulation → TRL 3-4
    - unknown → TRL 5 (middle ground)
    """
    validation = row.get('validation_type', 'unknown')
    
    trl_map = {
        'deployed': 8,
        'case_study': 6,
        'benchmark': 4,
        'simulation': 3,
        'unknown': 5
    }
    
    return trl_map.get(validation, 5)


def check_standards_citation(text: str) -> bool:
    """Check if paper cites standards (ISO 13374/23247, etc.)"""
    text_lower = text.lower()
    
    for pattern in CONFIG['standard_patterns']:
        if re.search(pattern, text_lower, re.IGNORECASE):
            return True
    
    return False


def check_dataset_availability(text: str) -> bool:
    """Check if paper mentions dataset availability"""
    text_lower = text.lower()
    
    return any(kw in text_lower for kw in CONFIG['dataset_keywords'])


def compute_reproducibility_score(row: pd.Series) -> float:
    """Compute reproducibility score (0-1) based on multiple factors
    
    Factors:
    - Dataset availability: 0.4
    - Open access: 0.3
    - Standards citation: 0.3
    """
    score = 0.0
    
    if row.get('has_dataset', False):
        score += 0.4
    
    if row.get('is_oa', False):
        score += 0.3
    
    if row.get('cites_standards', False):
        score += 0.3
    
    return round(score, 2)


def enrich_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich dataframe with gap-relevant metadata
    
    Adds columns:
    - validation_type: deployed/case_study/benchmark/simulation/unknown
    - application_domain: HVAC/Manufacturing/Energy/Infrastructure/Multi/General
    - trl_estimate: Technology Readiness Level (1-9)
    - cites_standards: Boolean
    - has_dataset: Boolean
    - reproducibility_score: Float (0-1)
    """
    logger.info("Enriching metadata with gap analysis fields...")
    
    # Validation type
    df['validation_type'] = df['text_for_nlp'].apply(classify_validation_type)
    
    # Application domain
    df['application_domain'] = df['text_for_nlp'].apply(classify_application_domain)
    
    # TRL estimate
    df['trl_estimate'] = df.apply(estimate_trl, axis=1)
    
    # Standards citation
    df['cites_standards'] = df['abstract'].apply(check_standards_citation)
    
    # Dataset availability
    df['has_dataset'] = df['text_for_nlp'].apply(check_dataset_availability)
    
    # Reproducibility score
    df['reproducibility_score'] = df.apply(compute_reproducibility_score, axis=1)
    
    # Log statistics
    logger.info(f"Validation types: {df['validation_type'].value_counts().to_dict()}")
    logger.info(f"Application domains: {df['application_domain'].value_counts().to_dict()}")
    logger.info(f"TRL range: {df['trl_estimate'].min()}-{df['trl_estimate'].max()}")
    logger.info(f"Standards citation: {df['cites_standards'].sum()} papers ({df['cites_standards'].mean()*100:.1f}%)")
    logger.info(f"Dataset availability: {df['has_dataset'].sum()} papers ({df['has_dataset'].mean()*100:.1f}%)")
    
    return df


# ===========================
# NETWORK CONSTRUCTION
# ===========================

def build_similarity_matrix(df: pd.DataFrame, 
                           max_features: int = 8000) -> np.ndarray:
    """Build TF-IDF similarity matrix for papers
    
    Args:
        df: DataFrame with 'text_for_nlp' column
        max_features: Maximum TF-IDF features
        
    Returns:
        Sparse similarity matrix (cosine)
    """
    logger.info(f"Computing TF-IDF similarity matrix ({len(df)} papers)...")
    
    tfidf = TfidfVectorizer(
        max_features=max_features,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.8
    )
    
    X = tfidf.fit_transform(df['text_for_nlp'].fillna(''))
    
    # Compute cosine similarity (sparse output for memory efficiency)
    similarity = cosine_similarity(X, dense_output=False)
    
    logger.info(f"Similarity matrix shape: {similarity.shape}")
    logger.info(f"Non-zero elements: {similarity.nnz}")
    
    return similarity


def compute_cooccurrence_matrix(df: pd.DataFrame, keywords: List[str]) -> pd.DataFrame:
    """Compute keyword co-occurrence matrix
    
    Args:
        df: DataFrame with 'kw_found' column (list of keywords per paper)
        keywords: List of all keywords
        
    Returns:
        Co-occurrence matrix as DataFrame
    """
    n = len(keywords)
    cooc_matrix = np.zeros((n, n), dtype=int)
    kw_to_idx = {kw: i for i, kw in enumerate(keywords)}
    
    for kw_list in df['kw_found']:
        if not isinstance(kw_list, list):
            continue
        
        # All pairs of keywords in this paper
        for kw1, kw2 in itertools.combinations(sorted(set(kw_list)), 2):
            if kw1 in kw_to_idx and kw2 in kw_to_idx:
                i, j = kw_to_idx[kw1], kw_to_idx[kw2]
                cooc_matrix[i, j] += 1
                cooc_matrix[j, i] += 1
    
    return pd.DataFrame(cooc_matrix, index=keywords, columns=keywords)


def build_network_graph(df: pd.DataFrame, 
                       similarity_matrix: np.ndarray,
                       keywords: List[str],
                       topic_colors: Dict[str, str],
                       sim_threshold: float,
                       min_degree: int) -> Tuple[nx.Graph, pd.DataFrame]:
    """Build network graph with topics and papers
    
    Args:
        df: Paper dataframe
        similarity_matrix: Paper-paper similarity matrix
        keywords: List of topic keywords
        topic_colors: Color mapping for topics
        sim_threshold: Minimum similarity for paper-paper edges
        min_degree: Minimum degree for paper nodes (pruning)
        
    Returns:
        (NetworkX graph, co-occurrence dataframe)
    """
    logger.info("Building network graph...")
    
    G = nx.Graph()
    
    # Add topic nodes on circle
    n_topics = len(keywords)
    radius = 2.8
    theta = np.linspace(0, 2 * np.pi, n_topics, endpoint=False)
    
    topic_nodes = []
    topic_positions = {}
    
    for i, keyword in enumerate(keywords):
        node_id = ('KW', keyword)
        topic_nodes.append(node_id)
        
        G.add_node(
            node_id,
            label=keyword,
            type='topic',
            color=topic_colors.get(keyword, '#888888'),
            size=1100
        )
        
        topic_positions[node_id] = (radius * np.cos(theta[i]), radius * np.sin(theta[i]))
    
    # Add paper nodes
    for idx, row in df.iterrows():
        node_id = ('P', int(idx))
        
        # Size based on citations (log scale)
        cited_by = safe_int(row['cited_by'], 0)
        size = 150 + 10 * np.sqrt(cited_by + 1)
        
        # Truncate long titles
        label = row['title']
        if len(label) > 120:
            label = label[:117] + '...'
        
        G.add_node(
            node_id,
            label=label,
            type='paper',
            color='#BBBBBB',  # Will be updated after community detection
            size=size,
            cited_by=cited_by,
            year=int(row['year']) if pd.notna(row['year']) else None,
            url=row.get('url'),
            venue=row.get('venue'),
            publisher=row.get('publisher'),
            source=row.get('source'),
            kw_found=row.get('kw_found', []),
            validation_type=row.get('validation_type'),
            application_domain=row.get('application_domain'),
            trl_estimate=row.get('trl_estimate'),
            reproducibility_score=row.get('reproducibility_score')
        )
        
        # Add paper-topic edges
        for keyword in row.get('kw_found', []):
            topic_node = ('KW', keyword)
            if topic_node in G:
                G.add_edge(
                    node_id,
                    topic_node,
                    kind='paper-topic',
                    weight=1.0,
                    color=topic_colors.get(keyword, '#888888')
                )
    
    # Add paper-paper edges based on similarity
    logger.info("Adding paper-paper similarity edges...")
    rows, cols = similarity_matrix.nonzero()
    
    edge_count = 0
    for r, c in zip(rows, cols):
        if r >= c:  # Only upper triangle
            continue
        
        sim_value = float(similarity_matrix[r, c])
        
        if sim_value >= sim_threshold:
            G.add_edge(
                ('P', int(r)),
                ('P', int(c)),
                kind='paper-paper',
                weight=sim_value,
                color='#8c8c8c'
            )
            edge_count += 1
    
    logger.info(f"Added {edge_count} paper-paper edges (threshold={sim_threshold})")
    
    # Pruning: remove low-degree paper nodes not connected to topics
    logger.info("Pruning low-degree nodes...")
    
    nodes_to_keep = set(topic_nodes)
    for node, degree in G.degree():
        if G.nodes[node].get('type') == 'paper':
            # Keep if: sufficient degree AND connected to at least one topic
            if degree >= min_degree and any(G.has_edge(node, t) for t in topic_nodes):
                nodes_to_keep.add(node)
    
    G = G.subgraph(nodes_to_keep).copy()
    
    # Keep only connected components containing topics
    components = list(nx.connected_components(G))
    keep_components = [comp for comp in components if any(n in topic_nodes for n in comp)]
    
    if keep_components:
        final_nodes = set().union(*keep_components)
        G = G.subgraph(final_nodes).copy()
    
    logger.info(f"After pruning: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    
    # Compute co-occurrence matrix
    cooc_df = compute_cooccurrence_matrix(df, keywords)
    
    return G, cooc_df, topic_positions


def detect_communities(G: nx.Graph) -> Dict:
    """Detect communities using Louvain algorithm
    
    Args:
        G: Network graph
        
    Returns:
        Dictionary mapping node to community ID
    """
    # Extract paper subgraph for community detection
    paper_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
    P = G.subgraph(paper_nodes).copy()
    
    if P.number_of_nodes() == 0:
        return {}
    
    logger.info(f"Detecting communities on {P.number_of_nodes()} papers...")
    
    partition = community_louvain.best_partition(
        P,
        random_state=CONFIG['seed'],
        weight='weight'
    )
    
    logger.info(f"Found {len(set(partition.values()))} communities")
    
    return partition


def assign_community_colors(G: nx.Graph, partition: Dict) -> nx.Graph:
    """Assign colors to papers based on community
    
    Args:
        G: Network graph
        partition: Node to community mapping
        
    Returns:
        Updated graph
    """
    if not partition:
        return G
    
    # Get unique community IDs
    unique_communities = sorted(set(partition.values()))
    
    # Generate colors using colormap
    cmap = plt.get_cmap('tab20')
    community_colors = {
        comm: cmap(i % 20)
        for i, comm in enumerate(unique_communities)
    }
    
    # Assign colors and community IDs to nodes
    for node, community_id in partition.items():
        if node in G:
            G.nodes[node]['group'] = community_id
            G.nodes[node]['color'] = community_colors.get(community_id, (0.7, 0.7, 0.7, 1.0))
    
    return G


# Continue in next part for layout and visualization...

In [4]:
# ===========================
# PART 4: ADVANCED VISUALIZATIONS
# ===========================

def compute_layout(G: nx.Graph, topic_positions: Dict) -> Dict:
    """Compute network layout with fixed topic positions
    
    Args:
        G: Network graph
        topic_positions: Pre-computed topic node positions
        
    Returns:
        Position dictionary
    """
    logger.info("Computing network layout...")
    
    # Fixed positions for topics
    fixed_nodes = list(topic_positions.keys())
    pos_init = {n: topic_positions[n] for n in fixed_nodes if n in G}
    
    # Spring layout for papers
    pos = nx.spring_layout(
        G,
        seed=CONFIG['seed'],
        k=0.7,
        iterations=180,
        weight='weight',
        pos=pos_init,
        fixed=fixed_nodes
    )
    
    return pos


def create_png_visualization(G: nx.Graph, pos: Dict, cooc_df: pd.DataFrame,
                            keywords: List[str], topic_colors: Dict[str, str],
                            output_file: str = 'slr_network_enhanced.png'):
    """Create enhanced static PNG visualization
    
    Features:
    - Improved color palette
    - Variable transparency for edges
    - Smart node sizing (log-scale for citations)
    - Top-paper annotations
    - Inset co-occurrence heatmap
    - Statistics info box
    - High-resolution export (300 DPI)
    """
    logger.info(f"Creating enhanced PNG visualization...")
    
    figsize = CONFIG['figsize']
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor('#ffffff')
    ax.axis('off')
    
    # Extract node lists
    topic_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'topic']
    paper_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
    
    # --- EDGES: Paper-Paper (with variable transparency) ---
    paper_paper_edges = [(u, v, e) for u, v, e in G.edges(data=True) if e.get('kind') == 'paper-paper']
    
    for u, v, edata in paper_paper_edges:
        weight = float(edata.get('weight', 1.0))
        sim_threshold = CONFIG['sim_threshold']
        
        # Width scaling
        width = max(0.3, min(2.2, 1.5 * (weight - sim_threshold) / (1.0 - sim_threshold)))
        
        # Alpha scaling (more weight = more visible)
        alpha = 0.15 + 0.35 * (weight - sim_threshold) / (1.0 - sim_threshold)
        
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=width,
            edge_color='#8c8c8c',
            alpha=alpha,
            style='solid'
        )
    
    # --- EDGES: Paper-Topic (colored by topic) ---
    paper_topic_edges = [(u, v, e) for u, v, e in G.edges(data=True) if e.get('kind') == 'paper-topic']
    
    for u, v, edata in paper_topic_edges:
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=1.2,
            edge_color=edata.get('color', '#888888'),
            alpha=0.55,
            style='solid'
        )
    
    # --- NODES: Papers (sized by citations, colored by community) ---
    paper_sizes = []
    paper_colors = []
    
    cited_by_95th = np.percentile([G.nodes[n]['cited_by'] for n in paper_nodes], 95)
    
    for node in paper_nodes:
        cited_by = G.nodes[node]['cited_by']
        
        # Smart sizing (log-bounded)
        base_size = 100
        max_size = 500
        size = base_size + (max_size - base_size) * (np.log1p(cited_by) / np.log1p(cited_by_95th))
        paper_sizes.append(size)
        
        # Color from community detection
        color = G.nodes[node].get('color', '#BBBBBB')
        paper_colors.append(color)
    
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=paper_nodes,
        node_size=paper_sizes,
        node_color=paper_colors,
        linewidths=0.3,
        edgecolors='#444444',
        alpha=0.95
    )
    
    # --- NODES: Topics (fixed size, colored squares) ---
    topic_colors_list = [G.nodes[n].get('color', '#888888') for n in topic_nodes]
    
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=topic_nodes,
        node_shape='s',
        node_color=topic_colors_list,
        node_size=1100,
        edgecolors='#222222',
        linewidths=1.4
    )
    
    # Topic labels (bold)
    topic_labels = {n: G.nodes[n]['label'] for n in topic_nodes}
    nx.draw_networkx_labels(
        G, pos,
        labels=topic_labels,
        font_size=12,
        font_weight='bold',
        font_color='white'
    )
    
    # --- ANNOTATIONS: Top-cited papers ---
    top_k = 10
    top_papers = sorted(paper_nodes, key=lambda n: G.nodes[n]['cited_by'], reverse=True)[:top_k]
    
    for node in top_papers:
        label = G.nodes[node]['label'][:30] + '...'
        x, y = pos[node]
        
        ax.annotate(
            label,
            xy=(x, y),
            xytext=(x + 0.15, y + 0.15),
            fontsize=7,
            alpha=0.8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#cccccc'),
            arrowprops=dict(arrowstyle='->', lw=0.5, color='#666666')
        )
    
    # --- INSET: Co-occurrence heatmap ---
    ax_inset = inset_axes(ax, width="18%", height="18%", loc='lower right')
    
    im = ax_inset.imshow(cooc_df.values, cmap='Greens', aspect='auto')
    
    short_labels = [kw[:8] + '.' if len(kw) > 8 else kw for kw in keywords]
    ax_inset.set_xticks(range(len(keywords)))
    ax_inset.set_xticklabels(short_labels, fontsize=6, rotation=45, ha='right')
    ax_inset.set_yticks(range(len(keywords)))
    ax_inset.set_yticklabels(short_labels, fontsize=6)
    ax_inset.set_title('Co-occurrence', fontsize=7)
    
    # --- INFO BOX: Statistics ---
    n_papers = len(paper_nodes)
    n_pp_edges = len(paper_paper_edges)
    n_pt_edges = len(paper_topic_edges)
    
    avg_citations = np.mean([G.nodes[n]['cited_by'] for n in paper_nodes])
    year_min = min((G.nodes[n]['year'] for n in paper_nodes if G.nodes[n]['year']), default=0)
    year_max = max((G.nodes[n]['year'] for n in paper_nodes if G.nodes[n]['year']), default=0)
    
    n_communities = len(set(G.nodes[n].get('group', -1) for n in paper_nodes))
    
    stats_text = f"""Papers: {n_papers}
Paper-Paper: {n_pp_edges}
Paper-Topic: {n_pt_edges}
Avg Cites: {avg_citations:.1f}
Years: {year_min}–{year_max}
Communities: {n_communities}"""
    
    ax.text(
        0.02, 0.98,
        stats_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='#ffffcc', alpha=0.9, edgecolor='#888888')
    )
    
    # --- LEGEND: External (right side) ---
    handles = [mpatches.Patch(color='#BBBBBB', label='Papers (colored by community)')]
    
    for kw in keywords:
        if ('KW', kw) in G:
            handles.append(mpatches.Patch(color=topic_colors.get(kw, '#888888'), label=f'Topic: {kw}'))
    
    legend = plt.legend(
        handles=handles,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        frameon=True,
        fontsize=9,
        title='Legend'
    )
    legend.get_frame().set_alpha(0.95)
    
    # --- TITLE ---
    plt.title(
        'Systematic Literature Review Network Analysis\n'
        'Enhanced Visualization with Gap-Aware Metadata',
        fontsize=15,
        fontweight='bold',
        pad=20
    )
    
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    
    # --- EXPORT: High-resolution PNG + SVG ---
    dpi = CONFIG['png_dpi']
    
    plt.savefig(output_file, dpi=dpi, bbox_inches='tight', facecolor='white', edgecolor='none')
    logger.info(f"Saved PNG: {output_file} ({dpi} DPI)")
    
    # SVG for scalability
    svg_file = output_file.replace('.png', '.svg')
    plt.savefig(svg_file, format='svg', bbox_inches='tight')
    logger.info(f"Saved SVG: {svg_file}")
    
    plt.show()


def create_html_interactive(G: nx.Graph, pos: Dict, keywords: List[str],
                            topic_colors: Dict[str, str],
                            output_file: str = 'slr_network_interactive.html'):
    """Create interactive HTML visualization with PyVis
    
    Features:
    - Interactive zoom/pan
    - Hover tooltips with full metadata
    - Collapsible legend
    - Search functionality (via JS injection)
    - Physics toggle button
    - Mobile-responsive CSS
    """
    if not HAS_PYVIS:
        logger.warning("PyVis not available, skipping HTML visualization")
        return
    
    logger.info("Creating interactive HTML visualization...")
    
    net = Network(
        height=CONFIG['html_height'],
        width=CONFIG['html_width'],
        bgcolor='#ffffff',
        font_color='#222222',
        notebook=False,
        directed=False
    )
    
    # Physics and interaction options
    net.set_options("""{
      "nodes": {
        "font": { "size": 16, "face": "Arial" }
      },
      "edges": {
        "smooth": { "enabled": false }
      },
      "physics": {
        "enabled": false
      },
      "interaction": {
        "hover": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }""")
    
    # Add nodes
    for node, data in G.nodes(data=True):
        node_id = str(node)
        
        # Build tooltip
        label_text = data.get('label', '')
        title_html = f"<b>{html.escape(label_text)}</b>"
        
        if data.get('type') == 'paper':
            meta_items = []
            
            if data.get('venue'):
                meta_items.append(f"Venue: {html.escape(str(data['venue']))}")
            
            if data.get('publisher'):
                meta_items.append(f"Publisher: {html.escape(str(data['publisher']))}")
            
            if data.get('year'):
                meta_items.append(f"Year: {data['year']}")
            
            meta_items.append(f"Citations: {data.get('cited_by', 0)}")
            
            if data.get('validation_type'):
                meta_items.append(f"Validation: {data['validation_type']}")
            
            if data.get('application_domain'):
                meta_items.append(f"Domain: {data['application_domain']}")
            
            if data.get('trl_estimate'):
                meta_items.append(f"TRL: {data['trl_estimate']}")
            
            if data.get('reproducibility_score'):
                meta_items.append(f"Reproducibility: {data['reproducibility_score']:.2f}")
            
            if data.get('url'):
                meta_items.append(f"<a href='{html.escape(data['url'])}' target='_blank'>Open Link</a>")
            
            title_html += "<br>" + "<br>".join(meta_items)
        
        # Node size
        raw_size = data.get('size', 150)
        if raw_size is None or (isinstance(raw_size, float) and np.isnan(raw_size)):
            raw_size = 150
        
        node_size = 26 if data.get('type') == 'topic' else max(6, int(raw_size / 18))
        
        # Node color
        color = data.get('color')
        if not isinstance(color, str):
            # Convert matplotlib color tuple to hex
            if isinstance(color, (tuple, list)) and len(color) >= 3:
                r, g, b = color[:3]
                color = f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'
            else:
                color = '#cccccc'
        
        net.add_node(
            node_id,
            label=data.get('label') if data.get('type') == 'topic' else '',
            title=title_html,
            color=color,
            shape='box' if data.get('type') == 'topic' else 'dot',
            size=node_size,
            x=pos[node][0] * 140,
            y=pos[node][1] * 140,
            physics=False
        )
    
    # Add edges
    sim_threshold = CONFIG['sim_threshold']
    
    for u, v, edata in G.edges(data=True):
        color = edata.get('color', '#8c8c8c')
        
        if edata.get('kind') == 'paper-paper':
            weight = float(edata.get('weight', 1.0))
            width = max(0.3, min(2.2, 1.5 * (weight - sim_threshold) / (1.0 - sim_threshold)))
        else:
            width = 1.2
        
        net.add_edge(str(u), str(v), color=color, width=width)
    
    # Generate base HTML
    base_html = net.generate_html()
    
    # Inject custom CSS and JS
    legend_items = "".join(
        f"<div class='legitem' style='background:{topic_colors[kw]};'>{html.escape('Topic: ' + kw)}</div>"
        for kw in keywords if ('KW', kw) in G
    )
    
    custom_html = f"""
<style>
/* Mobile responsive */
@media (max-width: 768px) {{
  #mynetwork {{ height: 600px !important; }}
  #slr-legend {{ font-size: 11px; padding: 6px; }}
}}

/* Legend */
#slr-legend {{
  position: fixed;
  right: 18px;
  bottom: 18px;
  z-index: 9999;
  background: rgba(255, 255, 255, 0.95);
  border: 1px solid #ccc;
  border-radius: 8px;
  padding: 10px 12px;
  font-family: Arial, sans-serif;
  font-size: 13px;
  color: #222;
  box-shadow: 0 2px 8px rgba(0, 0, 0, 0.15);
  max-height: 400px;
  overflow-y: auto;
}}

#slr-legend .title {{
  font-weight: bold;
  margin-bottom: 6px;
  cursor: pointer;
}}

#slr-legend .title::after {{
  content: ' ▼';
  float: right;
  font-size: 10px;
}}

#slr-legend.collapsed .title::after {{
  content: ' ▲';
}}

#slr-legend.collapsed .content {{
  display: none;
}}

#slr-legend .paper {{
  background: #BBBBBB;
  color: #222;
  padding: 4px 8px;
  border-radius: 6px;
  margin: 3px 0;
}}

#slr-legend .legitem {{
  color: #fff;
  padding: 4px 8px;
  border-radius: 6px;
  margin: 3px 0;
}}

/* Control buttons */
#controls {{
  position: fixed;
  top: 18px;
  right: 18px;
  z-index: 9999;
}}

#controls button {{
  margin: 4px;
  padding: 8px 12px;
  background: #4CAF50;
  color: white;
  border: none;
  border-radius: 4px;
  cursor: pointer;
  font-size: 13px;
}}

#controls button:hover {{
  background: #45a049;
}}
</style>

<div id="controls">
  <button id="physicsBtn" onclick="togglePhysics()">Physics: OFF</button>
  <button onclick="resetView()">Reset View</button>
</div>

<div id="slr-legend">
  <div class="title" onclick="toggleLegend()">Legend</div>
  <div class="content">
    <div class="paper">Papers (colored by community)</div>
    {legend_items}
  </div>
</div>

<script>
function toggleLegend() {{
  document.getElementById('slr-legend').classList.toggle('collapsed');
}}

function togglePhysics() {{
  const physEnabled = network.physics.options.enabled;
  network.setOptions({{ physics: {{ enabled: !physEnabled }} }});
  document.getElementById('physicsBtn').innerText = physEnabled ? 'Physics: OFF' : 'Physics: ON';
}}

function resetView() {{
  network.fit();
}}
</script>
"""
    
    final_html = base_html.replace('</body>', custom_html + '\n</body>')
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(final_html)
    
    logger.info(f"Saved HTML: {output_file}")


# Continue in next part for gap visualizations and exports...

In [5]:
# ===========================
# PART 5: GAP-AWARE VISUALIZATIONS & EXPORTS
# ===========================

def create_trl_distribution_plot(df: pd.DataFrame, output_file: str = 'slr_trl_distribution.png'):
    """Create TRL distribution visualization highlighting academic vs industrial gap"""
    logger.info("Creating TRL distribution plot...")
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    trl_counts = df['trl_estimate'].value_counts().sort_index()
    
    # Color code: red=academic, orange=transition, green=industrial
    colors = [
        '#ff4444' if x <= 3 else '#ffaa44' if x <= 6 else '#44ff44'
        for x in trl_counts.index
    ]
    
    bars = ax.bar(trl_counts.index, trl_counts.values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.2)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f'{int(height)}',
            ha='center',
            va='bottom',
            fontsize=10,
            fontweight='bold'
        )
    
    # Background zones
    ax.axvspan(1, 4, alpha=0.1, color='red', label='Academic (TRL 1-4)')
    ax.axvspan(5, 7, alpha=0.1, color='orange', label='Transition (TRL 5-7)')
    ax.axvspan(8, 9, alpha=0.1, color='green', label='Industrial (TRL 8-9)')
    
    ax.set_xlabel('Technology Readiness Level (TRL)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Papers', fontsize=12, fontweight='bold')
    ax.set_title(
        'TRL Distribution - Academic vs Industrial Gap Analysis',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.set_xticks(range(1, 10))
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    logger.info(f"Saved TRL plot: {output_file}")
    plt.show()


def create_domain_validation_matrix(df: pd.DataFrame, 
                                    output_file: str = 'slr_domain_validation_matrix.png'):
    """Create heatmap showing domain vs validation type - identifying white spaces"""
    logger.info("Creating domain-validation matrix...")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    pivot = pd.crosstab(df['application_domain'], df['validation_type'])
    
    # Reorder for better visibility
    domain_order = ['HVAC', 'Energy', 'Manufacturing', 'Infrastructure', 'Multi', 'General']
    validation_order = ['deployed', 'case_study', 'benchmark', 'simulation', 'unknown']
    
    pivot = pivot.reindex(index=domain_order, columns=validation_order, fill_value=0)
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Number of Papers'},
        linewidths=0.5,
        linecolor='gray'
    )
    
    ax.set_title(
        'Application Domain vs Validation Type\nIdentifying Research Gaps and White Spaces',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.set_xlabel('Validation Type', fontsize=12, fontweight='bold')
    ax.set_ylabel('Application Domain', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    logger.info(f"Saved domain-validation matrix: {output_file}")
    plt.show()


def create_standards_adoption_timeline(df: pd.DataFrame,
                                       output_file: str = 'slr_standards_adoption.png'):
    """Create timeline showing standards citation rate over years"""
    logger.info("Creating standards adoption timeline...")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Filter papers with valid years
    df_with_year = df[df['year'].notna()].copy()
    
    standards_by_year = df_with_year[df_with_year['cites_standards']].groupby('year').size()
    all_by_year = df_with_year.groupby('year').size()
    
    adoption_rate = (standards_by_year / all_by_year * 100).fillna(0)
    
    # Plot line with markers
    ax.plot(
        adoption_rate.index,
        adoption_rate.values,
        marker='o',
        linewidth=2.5,
        markersize=8,
        color='#2ca02c',
        label='Standards Citation Rate (%)'
    )
    
    # Fill area under curve
    ax.fill_between(adoption_rate.index, 0, adoption_rate.values, alpha=0.3, color='#2ca02c')
    
    # Add trend line
    if len(adoption_rate) > 2:
        z = np.polyfit(adoption_rate.index, adoption_rate.values, 1)
        p = np.poly1d(z)
        ax.plot(
            adoption_rate.index,
            p(adoption_rate.index),
            linestyle='--',
            color='#d62728',
            linewidth=2,
            alpha=0.7,
            label=f'Trend: {z[0]:+.2f}% per year'
        )
    
    ax.set_xlabel('Year', fontsize=12, fontweight='bold')
    ax.set_ylabel('Citation Rate (%)', fontsize=12, fontweight='bold')
    ax.set_title(
        'ISO 13374/23247 Standards Citation Trend\nStandardization Gap Analysis',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim(0, max(adoption_rate.values) * 1.2)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    logger.info(f"Saved standards adoption plot: {output_file}")
    plt.show()


def create_reproducibility_scatter(df: pd.DataFrame,
                                   output_file: str = 'slr_reproducibility_scatter.png'):
    """Create scatter plot of citations vs reproducibility score"""
    logger.info("Creating reproducibility scatter plot...")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Color by domain
    domains = df['application_domain'].unique()
    domain_colors = plt.cm.Set3(np.linspace(0, 1, len(domains)))
    color_map = dict(zip(domains, domain_colors))
    
    for domain in domains:
        subset = df[df['application_domain'] == domain]
        ax.scatter(
            subset['reproducibility_score'],
            subset['cited_by'],
            c=[color_map[domain]],
            label=domain,
            s=100,
            alpha=0.6,
            edgecolors='black',
            linewidth=0.5
        )
    
    ax.set_xlabel('Reproducibility Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Citation Count', fontsize=12, fontweight='bold')
    ax.set_title(
        'Citations vs Reproducibility Score by Domain\nHigher Reproducibility → Higher Impact?',
        fontsize=14,
        fontweight='bold',
        pad=15
    )
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_yscale('log')  # Log scale for citations
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    logger.info(f"Saved reproducibility scatter: {output_file}")
    plt.show()


# ===========================
# EXPORT FUNCTIONS
# ===========================

def export_latex_table(df: pd.DataFrame, output_file: str = 'slr_top_papers.tex',
                      top_n: int = 20):
    """Export top papers as LaTeX table"""
    logger.info(f"Exporting top {top_n} papers to LaTeX...")
    
    df_sorted = df.nlargest(top_n, 'cited_by')
    
    latex = r"""\begin{table}[h]
\centering
\caption{Top Papers by Citation Count}
\label{tab:top_papers}
\begin{tabular}{p{5cm}p{3cm}ccr}
\toprule
Title & Authors & Year & Venue & Citations \\
\midrule
"""
    
    for _, row in df_sorted.iterrows():
        title = row['title'][:50] + '...' if len(row['title']) > 50 else row['title']
        title = title.replace('&', r'\&').replace('_', r'\_').replace('%', r'\%')
        
        authors_str = row['authors']
        if isinstance(authors_str, str) and '; ' in authors_str:
            first_author = authors_str.split('; ')[0] + ' et al.'
        else:
            first_author = str(authors_str)[:30]
        first_author = first_author.replace('&', r'\&').replace('_', r'\_')
        
        year = row['year'] if pd.notna(row['year']) else 'N/A'
        
        venue = str(row['venue'])[:20] if pd.notna(row['venue']) else 'N/A'
        venue = venue.replace('&', r'\&').replace('_', r'\_')
        
        cited_by = row['cited_by']
        
        latex += f"{title} & {first_author} & {year} & {venue} & {cited_by} \\\\\n"
    
    latex += r"""\bottomrule
\end{tabular}
\end{table}
"""
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(latex)
    
    logger.info(f"Saved LaTeX table: {output_file}")


def export_bibtex(df: pd.DataFrame, output_file: str = 'slr_references.bib'):
    """Export papers as BibTeX entries"""
    logger.info("Exporting BibTeX references...")
    
    bib_entries = []
    
    for _, row in df.iterrows():
        if not row.get('doi'):
            continue
        
        # Generate cite key
        authors_str = row.get('authors', '')
        if isinstance(authors_str, str) and authors_str:
            first_author = authors_str.split('; ')[0].split()[-1] if '; ' in authors_str else authors_str.split()[-1]
        else:
            first_author = 'Unknown'
        
        year = row.get('year', 'XXXX')
        cite_key = f"{first_author}{year}".replace(' ', '')
        
        # Entry type
        entry_type = 'article' if row.get('venue') else 'misc'
        
        entry = f"@{entry_type}{{{cite_key},\n"
        entry += f"  title = {{{row['title']}}},\n"
        
        if isinstance(row.get('authors'), str):
            entry += f"  author = {{{row['authors']}}},\n"
        
        if pd.notna(row.get('year')):
            entry += f"  year = {{{int(row['year'])}}},\n"
        
        if row.get('venue'):
            entry += f"  journal = {{{row['venue']}}},\n"
        
        if row.get('doi'):
            entry += f"  doi = {{{row['doi']}}},\n"
        
        if row.get('url'):
            entry += f"  url = {{{row['url']}}},\n"
        
        entry += "}\n\n"
        
        bib_entries.append(entry)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.writelines(bib_entries)
    
    logger.info(f"Saved {len(bib_entries)} BibTeX entries: {output_file}")

def export_gexf(G: nx.Graph, output_file: str = 'slr_network.gexf'):
    """Export network to GEXF format for Gephi.
    
    FIX (MOD-1): NetworkX GEXF writer interprets Python list-typed node
    attributes as dynamic time-series data and tries to unpack each element
    as (value, start, end).  If the list contains plain strings (e.g. kw_found)
    the unpack fails with ValueError: too many values to unpack (expected 3).
    Solution: convert every list attribute to a semicolon-delimited string
    and replace None values with empty strings before serialising.
    """
    logger.info("Exporting network to GEXF...")
    G_export = G.copy()
    
    for node, data in G_export.nodes(data=True):
        # ── MOD-1 FIX: sanitise attribute types for GEXF compatibility ──
        for attr_key in list(data.keys()):
            val = data.get(attr_key)
            if isinstance(val, list):
                # Convert list → semicolon-delimited string
                G_export.nodes[node][attr_key] = '; '.join(str(v) for v in val)
            elif val is None:
                # Replace None → empty string (GEXF cannot serialise NoneType)
                G_export.nodes[node][attr_key] = ''
        
        # ── Convert matplotlib RGBA tuples → GEXF viz dict ──
        color = G_export.nodes[node].get('color')
        if isinstance(color, (tuple, list)) and len(color) >= 3:
            r, g, b = color[:3]
            G_export.nodes[node]['viz'] = {
                'color': {'r': int(r * 255), 'g': int(g * 255), 'b': int(b * 255)}
            }
            del G_export.nodes[node]['color']
    
    nx.write_gexf(G_export, output_file)
    logger.info(f"Saved GEXF: {output_file}")




def export_summary_stats(df: pd.DataFrame, G: nx.Graph, output_file: str = 'slr_summary_stats.txt'):
    """Export comprehensive summary statistics"""
    logger.info("Generating summary statistics...")
    
    paper_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
    
    stats = f"""
================================================
SYSTEMATIC LITERATURE REVIEW - SUMMARY STATISTICS
================================================

DATASET OVERVIEW
----------------
Total Papers: {len(df)}
Date Range: {df['year'].min():.0f} - {df['year'].max():.0f}
Sources: {', '.join(df['source'].unique())}

CITATION STATISTICS
-------------------
Total Citations: {df['cited_by'].sum()}
Average Citations per Paper: {df['cited_by'].mean():.2f}
Median Citations: {df['cited_by'].median():.0f}
Max Citations: {df['cited_by'].max()}
Papers with >100 citations: {(df['cited_by'] > 100).sum()}

NETWORK STATISTICS
------------------
Nodes (Papers): {len(paper_nodes)}
Nodes (Topics): {len([n for n, d in G.nodes(data=True) if d.get('type') == 'topic'])}
Total Edges: {G.number_of_edges()}
Paper-Paper Edges: {len([(u,v) for u,v,e in G.edges(data=True) if e.get('kind')=='paper-paper'])}
Paper-Topic Edges: {len([(u,v) for u,v,e in G.edges(data=True) if e.get('kind')=='paper-topic'])}
Average Degree: {np.mean([d for n, d in G.degree()]):.2f}
Network Density: {nx.density(G):.4f}

GAP ANALYSIS METRICS
--------------------
Validation Types:
{df['validation_type'].value_counts().to_string()}

Application Domains:
{df['application_domain'].value_counts().to_string()}

TRL Distribution:
{df['trl_estimate'].value_counts().sort_index().to_string()}

Standards Citation Rate: {df['cites_standards'].mean()*100:.1f}%
Dataset Availability Rate: {df['has_dataset'].mean()*100:.1f}%

Average Reproducibility Score: {df['reproducibility_score'].mean():.2f}

TOP 10 MOST CITED PAPERS
-------------------------
"""
    
    top_10 = df.nlargest(10, 'cited_by')[['title', 'year', 'cited_by']]
    for i, (_, row) in enumerate(top_10.iterrows(), 1):
        stats += f"{i}. {row['title'][:70]}... ({row['year']:.0f}) - {row['cited_by']} citations\n"
    
    stats += "\n================================================\n"
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(stats)
    
    logger.info(f"Saved summary stats: {output_file}")
    
    print(stats)


# Continue in final part for main script...

In [ ]:
# ===========================
# PART 6: MAIN ORCHESTRATION
# ===========================

def main(config_file: Optional[str] = None,
         output_dir: str = './output_slr',
         skip_fetch: bool = False,
         skip_viz: bool = False,
         user_overrides: Optional[Dict[str, Any]] = None):   # <-- FIX-A: nuovo parametro
    """Main execution pipeline
    
    Args:
        config_file:    Path to YAML config file
        output_dir:     Output directory for all results
        skip_fetch:     Skip API fetching (use cached data)
        skip_viz:       Skip visualizations (faster for data-only runs)
        user_overrides: Dict of config values to apply AFTER load_config().
                        Prevents load_config() from overwriting interactive
                        inputs (keywords, years_back, topic_colors, etc.).
                        Passed automatically by the CLI execution block.
    """
    global CONFIG
    
    # Load base configuration from YAML or defaults
    CONFIG = load_config(config_file)
    # FIX-A: apply user interactive overrides AFTER load_config()
    # Without this block, load_config() resets CONFIG to DEFAULT_CONFIG.copy()
    # and all keywords / dates entered interactively are silently lost.
    if user_overrides:
        CONFIG.update(user_overrides)
        logger.info(f"Applied user overrides: { {k: v for k, v in user_overrides.items() if k != 'topic_colors'} }")

    
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create cache directory
    Path(CONFIG['cache_dir']).mkdir(parents=True, exist_ok=True)
    
    logger.info("=" * 60)
    logger.info("SLR NETWORK ANALYSIS v2.0 - ENHANCED EDITION")
    logger.info("=" * 60)
    logger.info(f"Output directory: {output_dir.absolute()}")
    logger.info(f"Keywords: {CONFIG['keywords']}")
    logger.info(f"Years back: {CONFIG['years_back']}")
    logger.info(f"Max per keyword: {CONFIG['max_per_keyword']}")
    
    # ============================================
    # PHASE 1: DATA COLLECTION
    # ============================================
    if not skip_fetch:
        logger.info("\n" + "=" * 60)
        logger.info("PHASE 1: DATA COLLECTION FROM APIs")
        logger.info("=" * 60)
        
        all_records = []
        keywords = CONFIG['keywords']
        
        # arXiv
        logger.info("\n>>> Querying arXiv...")
        for kw in tqdm(keywords, desc="arXiv"):
            try:
                results = search_arxiv(kw, max_results=CONFIG['max_per_keyword'])
                all_records.extend(results)
            except Exception as e:
                logger.error(f"arXiv error for '{kw}': {e}")
        
        # OpenAlex
        logger.info("\n>>> Querying OpenAlex...")
        for kw in tqdm(keywords, desc="OpenAlex"):
            try:
                results = search_openalex(kw, per_page=50, max_pages=1)
                all_records.extend(results)
            except Exception as e:
                logger.error(f"OpenAlex error for '{kw}': {e}")
        
        # Semantic Scholar
        logger.info("\n>>> Querying Semantic Scholar...")
        for kw in tqdm(keywords, desc="SemanticScholar"):
            try:
                results = search_semanticscholar(kw, limit=CONFIG['max_per_keyword'])
                all_records.extend(results)
            except Exception as e:
                logger.error(f"Semantic Scholar error for '{kw}': {e}")
        
        # Crossref
        logger.info("\n>>> Querying Crossref (top publishers)...")
        for kw in tqdm(keywords, desc="Crossref"):
            try:
                results = search_crossref_publishers(kw, rows=CONFIG['max_per_keyword'] // 2)
                all_records.extend(results)
            except Exception as e:
                logger.error(f"Crossref error for '{kw}': {e}")
        
        logger.info(f"\nTotal records fetched: {len(all_records)}")
        
        # Deduplication
        logger.info("\n>>> Deduplicating records...")
        if HAS_FUZZY:
            all_records = fuzzy_dedup(all_records, threshold=90)
        else:
            all_records = exact_dedup(all_records)
        
        logger.info(f"After deduplication: {len(all_records)} unique papers")
        
    else:
        logger.info("Skipping API fetch (using cached data)")
        # Load from previous run if exists
        csv_path = output_dir / 'slr_results_all.csv'
        if csv_path.exists():
            logger.info(f"Loading existing data from {csv_path}")
            df = pd.read_csv(csv_path)
            all_records = df.to_dict('records')
        else:
            logger.error("No cached data found! Run without --skip-fetch first.")
            return
    
    # ============================================
    # PHASE 2: DATA PREPROCESSING
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("PHASE 2: DATA PREPROCESSING")
    logger.info("=" * 60)
    
    # Create dataframe
    df = pd.DataFrame(all_records)
    
    # Filter empty titles
    df = df[df['title'].str.len() > 0].copy()
    
    # ── Supplemental (MOD-2): apply end-year filter if user specified one ──
    if CONFIG.get('end_year') and CONFIG['end_year'] < datetime.now().year:
        _before = len(df)
        df = df[df['year'].fillna(0).astype(int) <= CONFIG['end_year']].copy()
        logger.info(f"End-year filter ({CONFIG['end_year']}): {_before} → {len(df)} papers")
    
    # Normalize fields
    df['abstract'] = df['abstract'].fillna('')

    
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['cited_by'] = pd.to_numeric(df['cited_by'], errors='coerce').fillna(0).astype(int)
    
    # Convert author lists to strings
    df['authors'] = df['authors'].apply(
        lambda a: '; '.join(a) if isinstance(a, list) else (a or '')
    )
    
    df['publisher'] = df['publisher'].fillna('')
    
    # Create NLP text
    df['text_for_nlp'] = (df['title'].fillna('') + '. ' + df['abstract'].fillna('')).str.strip()
    
    # Find keywords
    df['kw_found'] = df['text_for_nlp'].str.lower().apply(
        lambda x: match_keywords(x, CONFIG['keywords'])
    )
    
    df.reset_index(drop=True, inplace=True)
    
    logger.info(f"Preprocessed {len(df)} papers")
    
    # ============================================
    # PHASE 3: METADATA ENRICHMENT (GAP ANALYSIS)
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("PHASE 3: METADATA ENRICHMENT (GAP ANALYSIS)")
    logger.info("=" * 60)
    
    df = enrich_metadata(df)
    
    # ============================================
    # PHASE 4: EXPORT ENRICHED DATA
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("PHASE 4: EXPORTING ENRICHED DATA")
    logger.info("=" * 60)
    
    # Main CSV
    csv_out = output_dir / 'slr_results_all.csv'
    
    df_export = df[[
        'id', 'doi', 'title', 'authors', 'year', 'venue', 'publisher', 'source', 'url', 
        'cited_by', 'kw_found', 'validation_type', 'application_domain', 'trl_estimate',
        'cites_standards', 'has_dataset', 'reproducibility_score'
    ]].copy()
    
    df_export['kw_found'] = df_export['kw_found'].apply(
        lambda x: '; '.join(x) if isinstance(x, list) else ''
    )
    
    df_export.to_csv(csv_out, index=False)
    logger.info(f"Saved enriched CSV: {csv_out}")
    
    # ============================================
    # PHASE 5: NETWORK CONSTRUCTION
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("PHASE 5: NETWORK CONSTRUCTION")
    logger.info("=" * 60)
    
    # Build similarity matrix
    similarity_matrix = build_similarity_matrix(df)
    
    # Build network graph
    G, cooc_df, topic_positions = build_network_graph(
        df,
        similarity_matrix,
        CONFIG['keywords'],
        CONFIG['topic_colors'],
        CONFIG['sim_threshold'],
        CONFIG['min_degree']
    )
    
    # Detect communities
    partition = detect_communities(G)
    G = assign_community_colors(G, partition)
    
    # Compute layout
    pos = compute_layout(G, topic_positions)
    
    # Export edges CSV
    edges_csv = output_dir / 'slr_edges.csv'
    edges_rows = []
    for u, v, e in G.edges(data=True):
        edges_rows.append({
            'source_node': str(u),
            'target_node': str(v),
            'edge_type': e.get('kind'),
            'weight': float(e.get('weight', 1.0))
        })
    pd.DataFrame(edges_rows).to_csv(edges_csv, index=False)
    logger.info(f"Saved edges CSV: {edges_csv}")
    
    # ============================================
    # PHASE 6: VISUALIZATIONS
    # ============================================
    if not skip_viz:
        logger.info("\n" + "=" * 60)
        logger.info("PHASE 6: CREATING VISUALIZATIONS")
        logger.info("=" * 60)
        
        # Main network visualizations
        logger.info("\n>>> Creating enhanced PNG network visualization...")
        create_png_visualization(
            G, pos, cooc_df,
            CONFIG['keywords'],
            CONFIG['topic_colors'],
            output_file=str(output_dir / 'slr_network_enhanced.png')
        )
        
        logger.info("\n>>> Creating interactive HTML network...")
        create_html_interactive(
            G, pos,
            CONFIG['keywords'],
            CONFIG['topic_colors'],
            output_file=str(output_dir / 'slr_network_interactive.html')
        )
        
        # Gap-aware visualizations
        logger.info("\n>>> Creating gap analysis visualizations...")
        
        create_trl_distribution_plot(
            df,
            output_file=str(output_dir / 'slr_trl_distribution.png')
        )
        
        create_domain_validation_matrix(
            df,
            output_file=str(output_dir / 'slr_domain_validation_matrix.png')
        )
        
        create_standards_adoption_timeline(
            df,
            output_file=str(output_dir / 'slr_standards_adoption.png')
        )
        
        create_reproducibility_scatter(
            df,
            output_file=str(output_dir / 'slr_reproducibility_scatter.png')
        )
        
        # Co-occurrence heatmap (standalone)
        logger.info("\n>>> Creating co-occurrence heatmap...")
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cooc_df, annot=True, fmt='d', cmap='Greens', ax=ax, cbar_kws={'label': 'Co-occurrences'})
        ax.set_title('Topic Co-occurrence Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(output_dir / 'slr_topics_cooccurrence.png', dpi=300)
        plt.show()
        logger.info(f"Saved: {output_dir / 'slr_topics_cooccurrence.png'}")
    
    # ============================================
    # PHASE 7: EXPORTS
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("PHASE 7: GENERATING EXPORTS")
    logger.info("=" * 60)
    
    # LaTeX table
    export_latex_table(
        df,
        output_file=str(output_dir / 'slr_top_papers.tex'),
        top_n=20
    )
    
    # BibTeX
    export_bibtex(
        df,
        output_file=str(output_dir / 'slr_references.bib')
    )
    
    # GEXF for Gephi
    export_gexf(
        G,
        output_file=str(output_dir / 'slr_network.gexf')
    )
    
    # Summary statistics
    export_summary_stats(
        df, G,
        output_file=str(output_dir / 'slr_summary_stats.txt')
    )
    
    # ============================================
    # FINAL SUMMARY
    # ============================================
    logger.info("\n" + "=" * 60)
    logger.info("ANALYSIS COMPLETE!")
    logger.info("=" * 60)
    
    paper_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'paper']
    n_pp = len([(u, v) for u, v, e in G.edges(data=True) if e.get('kind') == 'paper-paper'])
    n_pt = len([(u, v) for u, v, e in G.edges(data=True) if e.get('kind') == 'paper-topic'])
    
    logger.info(f"""
Summary:
--------
Papers analyzed: {len(df)}
Papers in network: {len(paper_nodes)}
Paper-Paper edges: {n_pp}
Paper-Topic edges: {n_pt}
Communities detected: {len(set(partition.values())) if partition else 0}

Gap Analysis:
-------------
- TRL range: {df['trl_estimate'].min()}-{df['trl_estimate'].max()}
- Deployed validation: {(df['validation_type']=='deployed').sum()} papers
- Standards citation: {df['cites_standards'].sum()} papers ({df['cites_standards'].mean()*100:.1f}%)
- Dataset availability: {df['has_dataset'].sum()} papers ({df['has_dataset'].mean()*100:.1f}%)

Output Files:
-------------
All files saved to: {output_dir.absolute()}

Core outputs:
- slr_results_all.csv (enriched metadata)
- slr_edges.csv (network edges)
- slr_network_enhanced.png (high-res network)
- slr_network_enhanced.svg (scalable network)
- slr_network_interactive.html (interactive explore)

Gap visualizations:
- slr_trl_distribution.png
- slr_domain_validation_matrix.png
- slr_standards_adoption.png
- slr_reproducibility_scatter.png
- slr_topics_cooccurrence.png

Exports:
- slr_top_papers.tex (LaTeX table)
- slr_references.bib (BibTeX)
- slr_network.gexf (Gephi format)
- slr_summary_stats.txt (statistics)

Log file: slr_network.log
""")


# ===========================
# CLI ENTRY POINT
# ===========================
if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='SLR Network Analysis Tool v2.0 - Enhanced Edition',
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  # Basic run with default config
  python slr_network_v2_complete.py
  
  # Use custom config file
  python slr_network_v2_complete.py --config my_config.yaml
  
  # Skip API fetching (use cached data)
  python slr_network_v2_complete.py --skip-fetch
  
  # Skip visualizations (faster, data-only)
  python slr_network_v2_complete.py --skip-viz
  
  # Custom output directory
  python slr_network_v2_complete.py --output ./my_results
        """
    )
    
    parser.add_argument(
        '--config', '-c',
        type=str,
        default=None,
        help='Path to YAML configuration file'
    )
    
    parser.add_argument(
        '--output', '-o',
        type=str,
        default='./output_slr',
        help='Output directory for results (default: ./output_slr)'
    )
    
    parser.add_argument(
        '--skip-fetch',
        action='store_true',
        help='Skip API fetching, use cached data'
    )
    
    parser.add_argument(
        '--skip-viz',
        action='store_true',
        help='Skip visualizations (faster for data-only runs)'
    )
    #
    import sys
    sys.argv = ['']  # Simulate no command-line args in Jupyter
    args = parser.parse_args()
    
    # ================================================================
    # MOD-2: INTERACTIVE DATE RANGE INPUT
    # ================================================================
    # Replaces the static 'years_back' value in DEFAULT_CONFIG.
    # Both dates are validated before proceeding.
    # ================================================================
    print('\n' + '=' * 62)
    print('  SLR NETWORK ANALYSIS — PARAMETER CONFIGURATION')
    print('=' * 62)
    _default_start = f"{datetime.now().year - 5}-01-01"
    _default_end   = datetime.now().strftime('%Y-%m-%d')
    
    # --- Start date ---
    while True:
        _raw = input(f"  Start date YYYY-MM-DD  [default: {_default_start}]: ").strip()
        if not _raw:
            _raw = _default_start
        try:
            _start_dt  = datetime.strptime(_raw, '%Y-%m-%d')
            _start_year = _start_dt.year
            _start_str  = _raw
            break
        except ValueError:
            print('    ✗  Invalid format — use YYYY-MM-DD (e.g. 2019-01-15)')
    
    # --- End date ---
    while True:
        _raw = input(f"  End date   YYYY-MM-DD  [default: {_default_end}]: ").strip()
        if not _raw:
            _raw = _default_end
        try:
            _end_dt   = datetime.strptime(_raw, '%Y-%m-%d')
            _end_year = _end_dt.year
            _end_str  = _raw
            if _end_dt <= _start_dt:
                print(f'    ✗  End date must be after start date ({_start_str})')
                continue
            break
        except ValueError:
            print('    ✗  Invalid format — use YYYY-MM-DD')
    
    _years_back = datetime.now().year - _start_year
    
    # ================================================================
    # MOD-3: INTERACTIVE KEYWORD COUNT INPUT
    # ================================================================
    # Asks how many keywords the user wants to search for.
    # The count is used to drive the loop in MOD-4.
    # ================================================================
    while True:
        _raw = input('  Number of search keywords  [default: 5]: ').strip()
        if not _raw:
            _n_kw = 5
            break
        try:
            _n_kw = int(_raw)
            if _n_kw < 1:
                print('    ✗  Must be at least 1')
                continue
            if _n_kw > 20:
                print('    ⚠  More than 20 keywords may cause slow API queries. Continuing.')
            break
        except ValueError:
            print('    ✗  Enter a valid integer')
    
    # ================================================================
    # MOD-4: INTERACTIVE KEYWORD INPUT LOOP
    # ================================================================
    # Collects each keyword individually, validates non-empty.
    # Overwrites CONFIG['keywords'] and rebuilds CONFIG['topic_colors']
    # with a default palette so that any keyword gets a distinct colour.
    # ================================================================
    _kw_list = []
    print(f'\n  Enter {_n_kw} keyword(s), one per line:')
    for _i in range(1, _n_kw + 1):
        while True:
            _kw = input(f'    Keyword {_i}/{_n_kw}: ').strip()
            if _kw:
                _kw_list.append(_kw)
                break
            print('    ✗  Keyword cannot be empty — please re-enter')
    
    # ── Propagate user choices into CONFIG ──────────────────────────
    # ── Build colour palette for user keywords ──────────────────────
    # (palette is rebuilt here; direct CONFIG writes below are removed
    #  because load_config() inside main() would overwrite them anyway)
    _DEFAULT_PALETTE = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#aec7e8', '#ffbb78', '#98df8a', '#ff9896', '#c5b0d5',
        '#c49c94', '#f7b6d2', '#c7c7c7', '#dbdb8d', '#9edae5',
    ]
    _new_colors = {}
    for _idx, _kw in enumerate(_kw_list):
        _new_colors[_kw] = DEFAULT_CONFIG.get('topic_colors', {}).get(
            _kw, _DEFAULT_PALETTE[_idx % len(_DEFAULT_PALETTE)]
        )
    
    # ================================================================
    # FIX-B: raccoglie tutti i valori utente in _user_overrides e li
    # passa a main() come argomento esplicito.
    # main() applica CONFIG.update(user_overrides) DOPO load_config(),
    # garantendo che le keyword/date utente sopravvivano al reset di CONFIG.
    # ================================================================
    _user_overrides = {
        'keywords'    : _kw_list,
        'years_back'  : _years_back,
        'start_year'  : _start_year,
        'end_year'    : _end_year,
        'topic_colors': _new_colors,
    }
    
    # ── Summary confirmation ─────────────────────────────────────────
    print(f'\n  {"=" * 58}')
    print('  Configuration confirmed:')
    print(f'    Date range  : {_start_str}  →  {_end_str}')
    print(f'    Years back  : {_years_back}  (used as API filter)')
    print(f'    Keywords    : {_kw_list}')
    print(f'  {"=" * 58}\n')
    
    # ================================================================
    # MAIN EXECUTION
    # ================================================================
    try:
        main(
            config_file=args.config,
            output_dir=args.output,
            skip_fetch=args.skip_fetch,
            skip_viz=args.skip_viz,
            user_overrides=_user_overrides,   # <-- FIX-B: unica riga aggiunta
        )
    except Exception as e:
        logger.error(f"Fatal error: {e}", exc_info=True)
        raise

    
    



# ===========================
# MERGE INSTRUCTIONS
# ===========================
"""
TO CREATE COMPLETE SCRIPT:
--------------------------
Merge all parts in order:

cat slr_network_v2_enhanced_part1.py \\
    slr_network_v2_enhanced_part2.py \\
    slr_network_v2_enhanced_part3.py \\
    slr_network_v2_enhanced_part4.py \\
    slr_network_v2_enhanced_part5.py \\
    slr_network_v2_enhanced_part6.py > slr_network_v2_complete.py

DEPENDENCIES:
-------------
pip install requests pandas numpy scikit-learn networkx \\
    matplotlib seaborn python-louvain tqdm pyvis \\
    fuzzywuzzy python-Levenshtein pyyaml

USAGE:
------
python slr_network_v2_complete.py

For options:
python slr_network_v2_complete.py --help
"""


  SLR NETWORK ANALYSIS — PARAMETER CONFIGURATION
